# Data Cleaning
Clean and filter all three datasets, then save to `data/2_cleaned/`.
1. LGBT_EU:   or  
2. HIV_AIDS: 
3. UNICEF_Immunization: 


| Dataset | Files | Notes |
|---|---|---|
| `lgbt_EU` | 4 CSVs | A study on European Queer adults on various life experiences, like experiencing bigotry. The foundation of this analysis|
| `HIV_AIDS_data` | 6 CSVs (two schemas) | A study on HIV and AIDS prevalence, survivors, deaths, and experiences. Worldwide, will be filtered to EU only |
| `UNICEF_Immunization` | 1 xlsx, many vaccine sheets | A record of rates of immunization on a wide range of various vaccines.  Worldwide, will be filtered to EU only |

**Prerequisite:** `5_download_dataset.ipynb` — all raw files must exist in `data/1_source/`.

**Output:** `data/2_cleaned/` with one CSV per dataset (HIV/AIDS split by schema type).

## Setup

In [ ]:
# !pip install -r ../requirements.txt


     -------------------------------------- 250.9/250.9 kB 2.2 MB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import sys
import re
from pathlib import Path

import pandas as pd

In [18]:
# Ensure src/ is on the path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_download import *

CLEANED_DIR = PROJECT_ROOT / "data" / "2_cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Source dir   : {SOURCE_DIR}")
print(f"Cleaned dir  : {CLEANED_DIR}")

Project root : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG
Source dir   : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\1_source
Cleaned dir  : C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\2_cleaned


---
## 1. LGBT EU Survey

**Goals:**
- Load 4 CSVs (skip `SubsetSize`)
- Drop the `notes` column if present
- Remove rows where `CountryCode == 'Average'`
- Extract the canonical EU country list (used to filter the other datasets)
- Save one cleaned CSV per source file

In [19]:
LGBT_DIR = SOURCE_DIR / "lgbt_EU"

# Files to process — explicitly exclude SubsetSize
SKIP_LGBT = {"LGBT_Survey_SubsetSize.csv"}

lgbt_files = [
    f for f in sorted(LGBT_DIR.glob("*.csv"))
    if f.name not in SKIP_LGBT
]

print(f"LGBT files to clean: {len(lgbt_files)}")
for f in lgbt_files:
    print(f"  {f.name}")

LGBT files to clean: 5
  LGBT_Survey_DailyLife.csv
  LGBT_Survey_Discrimination.csv
  LGBT_Survey_RightsAwareness.csv
  LGBT_Survey_TransgenderSpecificQuestions.csv
  LGBT_Survey_ViolenceAndHarassment.csv


In [20]:
def clean_lgbt_csv(path):
    """
    Load and clean one LGBT survey CSV.
    - Drops 'notes' column if present
    - Removes rows where CountryCode == 'Average'
    - Strips leading/trailing whitespace from string columns
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape      : {df.shape}")
    print(f"  Columns        : {list(df.columns)}")

    # Drop 'notes' column
    if "notes" in df.columns:
        df = df.drop(columns=["notes"])
        print(f"  Dropped        : 'notes'")

    # Remove 'Average' rows
    before = len(df)
    df = df[df["CountryCode"] != "Average"]
    removed = before - len(df)
    print(f"  Removed 'Average' rows : {removed}")

    # Strip whitespace from all string columns
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

    print(f"  Clean shape    : {df.shape}")
    return df

In [21]:
lgbt_cleaned = {}

for f in lgbt_files:
    key = f.stem   # e.g. 'LGBT_Survey_ViolenceAndHarassment'
    lgbt_cleaned[key] = clean_lgbt_csv(f)


LGBT_Survey_DailyLife.csv
  Raw shape      : (34020, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 1184
  Clean shape    : (32836, 6)

LGBT_Survey_Discrimination.csv
  Raw shape      : (15775, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 547
  Clean shape    : (15228, 6)


C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns
C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m


LGBT_Survey_RightsAwareness.csv
  Raw shape      : (3770, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 130
  Clean shape    : (3640, 6)

LGBT_Survey_TransgenderSpecificQuestions.csv
  Raw shape      : (3421, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 120
  Clean shape    : (3301, 6)

LGBT_Survey_ViolenceAndHarassment.csv
  Raw shape      : (45355, 7)
  Columns        : ['CountryCode', 'subset', 'question_code', 'question_label', 'answer', 'percentage', 'notes']
  Dropped        : 'notes'
  Removed 'Average' rows : 1722
  Clean shape    : (43633, 6)


C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns
C:\Users\RAZER\AppData\Local\Temp\ipykernel_1012\1989910639.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

In [22]:
# Extract canonical EU country list from the survey data
# Use the first cleaned dataframe — all share the same CountryCode values
_sample_df = next(iter(lgbt_cleaned.values()))
EU_COUNTRIES: set[str] = set(_sample_df["CountryCode"].dropna().unique())

print(f"EU countries in survey ({len(EU_COUNTRIES)}):")
print(sorted(EU_COUNTRIES))

EU countries in survey (28):
['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czech Republic', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'United Kingdom']


In [23]:
# Inspect a sample of one cleaned dataframe
key = list(lgbt_cleaned.keys())[0]
print(f"Sample from: {key}")
lgbt_cleaned[key].head(6)

Sample from: LGBT_Survey_DailyLife


,CountryCode,subset,question_code,question_label,answer,percentage
0,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very widespread,8
1,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly widespread,34
2,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Fairly rare,45
3,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Very rare,9
4,Austria,Lesbian,b1_a,"In your opinion, how widespread is offensive l...",Don`t know,4
5,Austria,Gay,b1_a,"In your opinion, how widespread is offensive l...",Very widespread,4


In [24]:
# Save cleaned LGBT CSVs
lgbt_out_dir = CLEANED_DIR / "lgbt_EU"
lgbt_out_dir.mkdir(parents=True, exist_ok=True)

for key, df in lgbt_cleaned.items():
    out_path = lgbt_out_dir / f"{key}_cleaned.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"  Saved: {out_path.name}  ({len(df)} rows)")

  Saved: LGBT_Survey_DailyLife_cleaned.csv  (32836 rows)
  Saved: LGBT_Survey_Discrimination_cleaned.csv  (15228 rows)
  Saved: LGBT_Survey_RightsAwareness_cleaned.csv  (3640 rows)
  Saved: LGBT_Survey_TransgenderSpecificQuestions_cleaned.csv  (3301 rows)
  Saved: LGBT_Survey_ViolenceAndHarassment_cleaned.csv  (43633 rows)


In [25]:
LIST_OF_EU_COUNTRIES = [
    'Austria',
    'Belgium',
    'Bulgaria',
    'Croatia', 
    'Cyprus', 
    'Czech Republic', 
    'Denmark', 
    'Estonia', 
    'Finland', 
    'France', 
    'Germany', 
    'Greece', 
    'Hungary', 
    'Ireland', 
    'Italy', 
    'Latvia', 
    'Lithuania', 
    'Luxembourg', 
    'Malta', 
    'Netherlands', 
    'Poland', 
    'Portugal', 
    'Romania', 
    'Slovakia', 
    'Slovenia', 
    'Spain', 
    'Sweden', 
    'United Kingdom'
]

---
## 2. HIV/AIDS Data

Note, there are 2 different formats/schemas in this dataset:

**Schema A** — `no_of_<...>` files:  
Columns: `Country, Year, Count, Count_median, Count_min, Count_max, WHO Region`  
→ Drop `Count` since it uses a bracketed range string, keep `Count_median` as canonical.  
→ Normalize string nulls (`na`, `Na`, etc.) → `NaN`.

**Schema B** — `art_coverage_<...>` files:  
Columns: wide descriptive names, no `Year`, `Nodata` as string null, bracketed ranges in cells.  
→ Normalize `Nodata` / `No data` → `NaN`.  
→ Parse bracketed ranges (e.g. `500[500-500]`) — extract the leading number as the canonical value.

Both: filter to EU countries only.

In [26]:
HIV_DIR = SOURCE_DIR / "HIV_AIDS_data"

hiv_files = sorted(HIV_DIR.glob("*.csv"))
print(f"HIV/AIDS files: {len(hiv_files)}")
for f in hiv_files:
    print(f"  {f.name}")

HIV/AIDS files: 6
  art_coverage_by_country_clean.csv
  art_pediatric_coverage_by_country_clean.csv
  no_of_cases_adults_15_to_49_by_country_clean.csv
  no_of_deaths_by_country_clean.csv
  no_of_people_living_with_hiv_by_country_clean.csv
  prevention_of_mother_to_child_transmission_by_country_clean.csv


In [36]:
# Identify schema by filename prefix
SCHEMA_A_PREFIX = "no_of_"
SCHEMA_B_PREFIX = "art_"

schema_a_files = [f for f in hiv_files if f.name.startswith(SCHEMA_A_PREFIX)]
schema_b_files = [f for f in hiv_files if f.name.startswith(SCHEMA_B_PREFIX)]

print(f"Schema A (no_of_*): {[f.name for f in schema_a_files]}")
print(f"Schema B (art_*  ): {[f.name for f in schema_b_files]}")

Schema A (no_of_*): ['no_of_cases_adults_15_to_49_by_country_clean.csv', 'no_of_deaths_by_country_clean.csv', 'no_of_people_living_with_hiv_by_country_clean.csv']
Schema B (art_*  ): ['art_coverage_by_country_clean.csv', 'art_pediatric_coverage_by_country_clean.csv']


In [37]:
# --- Shared helper ---

NULL_STRINGS = {"na", "n/a", "nodata", "no data", "", "none", "-", ".."}

def normalize_nulls(df: pd.DataFrame) -> pd.DataFrame:
    """Replace common string null representations with actual NaN."""
    return df.replace(
        {v: pd.NA for v in NULL_STRINGS | {s.title() for s in NULL_STRINGS} | {s.upper() for s in NULL_STRINGS}}
    )


def extract_leading_number(val) -> float | None:
    """
    Extract the leading number from a string like '3500[3000-4400]' or '500[500-500]'.
    Returns the number as float, or None if not parseable.
    """
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    match = re.match(r'^([\d,\.]+)', val_str)
    if match:
        return float(match.group(1).replace(",", ""))
    return None

In [38]:
# --- Schema A: no_of_* files ---

def clean_hiv_schema_a(path: Path, eu_countries: set[str]) -> pd.DataFrame:
    """
    Clean a Schema A HIV file (no_of_* pattern).
    - Drop 'Count' (raw string with embedded ranges)
    - Keep Count_median as the canonical value
    - Normalize null strings
    - Filter to EU countries
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape  : {df.shape}")
    print(f"  Columns    : {list(df.columns)}")

    # Drop raw 'Count' column — median/min/max columns already exist
    if "Count" in df.columns:
        df = df.drop(columns=["Count"])
        print(f"  Dropped    : 'Count'")

    # Normalize null strings
    df = normalize_nulls(df)

    # Cast numeric columns
    for col in ["Count_median", "Count_min", "Count_max"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filter to EU countries
    before = len(df)
    df = df[df["Country"].isin(eu_countries)]
    print(f"  EU filter  : {before} → {len(df)} rows")

    return df


schema_a_cleaned: dict[str, pd.DataFrame] = {}

for f in schema_a_files:
    schema_a_cleaned[f.stem] = clean_hiv_schema_a(f, EU_COUNTRIES)


no_of_cases_adults_15_to_49_by_country_clean.csv
  Raw shape  : (680, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 680 → 104 rows

no_of_deaths_by_country_clean.csv
  Raw shape  : (510, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 510 → 78 rows

no_of_people_living_with_hiv_by_country_clean.csv
  Raw shape  : (680, 7)
  Columns    : ['Country', 'Year', 'Count', 'Count_median', 'Count_min', 'Count_max', 'WHO Region']
  Dropped    : 'Count'
  EU filter  : 680 → 104 rows


In [39]:
# Inspect Schema A sample
key = list(schema_a_cleaned.keys())[0]
print(f"Sample from: {key}")
schema_a_cleaned[key].head(6)

Sample from: no_of_cases_adults_15_to_49_by_country_clean


,Country,Year,Count_median,Count_min,Count_max,WHO Region
7,Austria,2018,NaN,NaN,NaN,Europe
14,Belgium,2018,NaN,NaN,NaN,Europe
23,Bulgaria,2018,0.1,0.1,0.1,Europe
39,Croatia,2018,0.1,0.1,0.1,Europe
41,Cyprus,2018,NaN,NaN,NaN,Europe
45,Denmark,2018,0.1,0.1,0.1,Europe


In [40]:
# --- Schema B: art_coverage_* files ---

def clean_hiv_schema_b(path: Path, eu_countries: set[str]) -> pd.DataFrame:
    """
    Clean a Schema B HIV file (art_coverage / art_pediatric pattern).
    - No 'Year' column — this is a snapshot dataset
    - Normalize 'Nodata' / 'No data' → NaN
    - Parse bracketed ranges like '500[500-500]' → extract leading number
    - Filter to EU countries
    """
    df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    print(f"\n{path.name}")
    print(f"  Raw shape  : {df.shape}")
    print(f"  Columns    : {list(df.columns)}")

    # Normalize null strings first
    df = normalize_nulls(df)

    # Parse bracketed ranges in all non-Country, non-WHO columns
    skip_cols = {"Country", "WHO Region"}
    for col in df.columns:
        if col in skip_cols:
            continue
        if df[col].dtype == object:
            has_brackets = df[col].dropna().astype(str).str.contains(r'\[', regex=True).any()
            if has_brackets:
                df[col] = df[col].apply(extract_leading_number)
                print(f"  Parsed brackets in: '{col}'")
            else:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    # Filter to EU countries
    before = len(df)
    df = df[df["Country"].isin(eu_countries)]
    print(f"  EU filter  : {before} → {len(df)} rows")

    return df


schema_b_cleaned: dict[str, pd.DataFrame] = {}

for f in schema_b_files:
    schema_b_cleaned[f.stem] = clean_hiv_schema_b(f, EU_COUNTRIES)


art_coverage_by_country_clean.csv
  Raw shape  : (170, 11)
  Columns    : ['Country', 'Reported number of people receiving ART', 'Estimated number of people living with HIV', 'Estimated ART coverage among people living with HIV (%)', 'Estimated number of people living with HIV_median', 'Estimated number of people living with HIV_min', 'Estimated number of people living with HIV_max', 'Estimated ART coverage among people living with HIV (%)_median', 'Estimated ART coverage among people living with HIV (%)_min', 'Estimated ART coverage among people living with HIV (%)_max', 'WHO Region']
  EU filter  : 170 → 26 rows

art_pediatric_coverage_by_country_clean.csv
  Raw shape  : (170, 11)
  Columns    : ['Country', 'Reported number of children receiving ART', 'Estimated number of children needing ART based on WHO methods', 'Estimated ART coverage among children (%)', 'Estimated number of children needing ART based on WHO methods_median', 'Estimated number of children needing ART based on WH

In [41]:
# Inspect Schema B sample
key = list(schema_b_cleaned.keys())[0]
print(f"Sample from: {key}")
schema_b_cleaned[key].head(6)

Sample from: art_coverage_by_country_clean


,Country,Reported number of people receiving ART,Estimated number of people living with HIV,Estimated ART coverage among people living with HIV (%),Estimated number of people living with HIV_median,Estimated number of people living with HIV_min,Estimated number of people living with HIV_max,Estimated ART coverage among people living with HIV (%)_median,Estimated ART coverage among people living with HIV (%)_min,Estimated ART coverage among people living with HIV (%)_max,WHO Region
7,Austria,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
14,Belgium,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
23,Bulgaria,1500,3500[3000–4100],41[35–48],3500.0,3000.0,4100.0,41.0,35.0,48.0,Europe
39,Croatia,1200,1600[1400–1700],75[67–83],1600.0,1400.0,1700.0,75.0,67.0,83.0,Europe
41,Cyprus,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe
45,Denmark,5500,6200[5600–7000],89[79–95],6200.0,5600.0,7000.0,89.0,79.0,95.0,Europe


In [42]:
# Save cleaned HIV/AIDS CSVs
hiv_out_dir = CLEANED_DIR / "HIV_AIDS_data"
hiv_out_dir.mkdir(parents=True, exist_ok=True)

for key, df in {**schema_a_cleaned, **schema_b_cleaned}.items():
    out_path = hiv_out_dir / f"{key}_cleaned.csv"
    df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"  Saved: {out_path.name}  ({len(df)} rows)")

  Saved: no_of_cases_adults_15_to_49_by_country_clean_cleaned.csv  (104 rows)
  Saved: no_of_deaths_by_country_clean_cleaned.csv  (78 rows)
  Saved: no_of_people_living_with_hiv_by_country_clean_cleaned.csv  (104 rows)
  Saved: art_coverage_by_country_clean_cleaned.csv  (26 rows)
  Saved: art_pediatric_coverage_by_country_clean_cleaned.csv  (26 rows)


---
## 3. UNICEF Immunization

The xlsx file has:
- A `README` sheet → skip
- One sheet per vaccine (BCG, DTP1, DTP2, HEPB3, ...)

Each vaccine sheet is wide-format with years as columns (2000–2023).

**Strategy:**
1. Read all vaccine sheets using `pd.read_excel`
2. Melt year columns → long format (`year`, `coverage_pct`)
3. Add a `vaccine` column tagging each row
4. Concatenate all sheets into one unified DataFrame
5. Filter to EU countries using the `country` column
6. Save as a single cleaned CSV

In [65]:
UNICEF_PATH = SOURCE_DIR / "UNICEF_Immunization" / "wuenic2023rev_web-update.xlsx"

print(f"UNICEF file: {UNICEF_PATH}")
print(f"Exists     : {'✅' if UNICEF_PATH.exists() else '❌'}")

# Peek at all sheet names
xl = pd.ExcelFile(UNICEF_PATH, engine="openpyxl")
all_sheets = xl.sheet_names
print(f"\nAll sheets ({len(all_sheets)}): {all_sheets}")

UNICEF file: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\1_source\UNICEF_Immunization\wuenic2023rev_web-update.xlsx
Exists     : ✅

All sheets (18): ['ReadMe', 'BCG', 'DTP1', 'DTP3', 'HEPB3', 'HEPBB', 'HIB3', 'IPV1', 'IPV2', 'MCV1', 'MCV2', 'MENGA', 'PCV3', 'POL3', 'RCV1', 'ROTAC', 'YFV', 'regional_global']


In [66]:
# Identify vaccine sheets — skip non-data sheets (case-insensitive)
SKIP_SHEETS = {"readme", "read me", "notes", "regional_global"}

vaccine_sheets = [s for s in all_sheets if s.lower().strip() not in SKIP_SHEETS]
print(f"Vaccine sheets to process ({len(vaccine_sheets)}): {vaccine_sheets}")

Vaccine sheets to process (16): ['BCG', 'DTP1', 'DTP3', 'HEPB3', 'HEPBB', 'HIB3', 'IPV1', 'IPV2', 'MCV1', 'MCV2', 'MENGA', 'PCV3', 'POL3', 'RCV1', 'ROTAC', 'YFV']


In [67]:
# Peek at one sheet to confirm the structure before processing all
sample_sheet = pd.read_excel(UNICEF_PATH, sheet_name=vaccine_sheets[0], engine="openpyxl")
print(f"Sheet '{vaccine_sheets[0]}' shape: {sample_sheet.shape}")
print(f"Columns: {list(sample_sheet.columns)}")
sample_sheet.head(3)

Sheet 'BCG' shape: (164, 28)
Columns: ['unicef_region', 'iso3', 'country', 'vaccine', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000']


,unicef_region,iso3,country,vaccine,2023,2022,2021,2020,2019,2018,...,2009,2008,2007,2006,2005,2004,2003,2002,2001,2000
0,ROSA,AFG,Afghanistan,BCG,68.0,69.0,65.0,72.0,74.0,82.0,...,64.0,66.0,60.0,60.0,57.0,51.0,44.0,46.0,43.0,30.0
1,ECAR,ALB,Albania,BCG,99.0,99.0,99.0,98.0,99.0,99.0,...,97.0,99.0,98.0,97.0,98.0,97.0,95.0,94.0,93.0,93.0
2,MENA,DZA,Algeria,BCG,99.0,98.0,98.0,99.0,99.0,99.0,...,99.0,99.0,99.0,99.0,98.0,98.0,98.0,98.0,97.0,97.0


In [68]:
def process_unicef_sheet(xl: pd.ExcelFile, sheet_name: str) -> pd.DataFrame:
    df = xl.parse(sheet_name)

    # Guard: skip sheets with no usable year columns
    year_cols = [c for c in df.columns if str(c).isdigit()]
    if df.empty or not year_cols:
        return pd.DataFrame()

    id_cols = [c for c in df.columns if not str(c).isdigit()]

    df_long = df.melt(
        id_vars=id_cols,
        value_vars=year_cols,
        var_name="year",
        value_name="coverage_pct",
    )

    df_long["year"] = df_long["year"].astype(int)
    df_long["coverage_pct"] = pd.to_numeric(df_long["coverage_pct"], errors="coerce")
    df_long["vaccine"] = sheet_name

    return df_long

In [69]:
unicef_frames = []

for sheet in vaccine_sheets:
    df_sheet = process_unicef_sheet(xl, sheet)
    unicef_frames.append(df_sheet)
    print(f"  {sheet:<15}  {len(df_sheet):>6} rows after melt")

unicef_long = pd.concat(unicef_frames, ignore_index=True)
print(f"\nCombined UNICEF long-format shape: {unicef_long.shape}")

  BCG                3936 rows after melt
  DTP1               4680 rows after melt
  DTP3               4680 rows after melt


  HEPB3              4584 rows after melt
  HEPBB              2616 rows after melt
  HIB3               4656 rows after melt
  IPV1               4680 rows after melt
  IPV2               2040 rows after melt
  MCV1               4680 rows after melt
  MCV2               4560 rows after melt
  MENGA               360 rows after melt
  PCV3               3792 rows after melt
  POL3               4680 rows after melt
  RCV1               4224 rows after melt
  ROTAC              3000 rows after melt
  YFV                 912 rows after melt

Combined UNICEF long-format shape: (58080, 6)


In [70]:
# Inspect the combined dataframe
print(f"Columns: {list(unicef_long.columns)}")
print(f"Vaccines: {sorted(unicef_long['vaccine'].unique())}")
unicef_long.head(6)

Columns: ['unicef_region', 'iso3', 'country', 'vaccine', 'year', 'coverage_pct']
Vaccines: ['BCG', 'DTP1', 'DTP3', 'HEPB3', 'HEPBB', 'HIB3', 'IPV1', 'IPV2', 'MCV1', 'MCV2', 'MENGA', 'PCV3', 'POL3', 'RCV1', 'ROTAC', 'YFV']


,unicef_region,iso3,country,vaccine,year,coverage_pct
0,ROSA,AFG,Afghanistan,BCG,2023,68.0
1,ECAR,ALB,Albania,BCG,2023,99.0
2,MENA,DZA,Algeria,BCG,2023,99.0
3,ESAR,AGO,Angola,BCG,2023,73.0
4,LACR,ARG,Argentina,BCG,2023,69.0
5,ECAR,ARM,Armenia,BCG,2023,99.0


In [71]:
# Filter to EU countries
# The UNICEF 'country' column uses full country names — same as the LGBT survey
before = len(unicef_long)
unicef_eu = unicef_long[unicef_long["country"].isin(EU_COUNTRIES)].copy()

print(f"UNICEF rows before EU filter : {before}")
print(f"UNICEF rows after  EU filter : {len(unicef_eu)}")
print(f"Countries retained           : {sorted(unicef_eu['country'].unique())}")

UNICEF rows before EU filter : 58080
UNICEF rows after  EU filter : 7008
Countries retained           : ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'United Kingdom']


In [ ]:
# # Check for any EU countries in the survey not found in UNICEF
# unicef_countries = set(unicef_eu["country"].unique())
# missing_from_unicef = EU_COUNTRIES - unicef_countries

# if missing_from_unicef:
#     print(f"⚠️  EU survey countries NOT found in UNICEF data ({len(missing_from_unicef)}):")
#     print(f"   {sorted(missing_from_unicef)}")
#     print("   These may use different country name spellings in the UNICEF dataset.")
# else:
#     print("✅  All EU survey countries found in UNICEF data.")

⚠️  EU survey countries NOT found in UNICEF data (2):
   ['Czech Republic', 'Netherlands']
   These may use different country name spellings in the UNICEF dataset.


In [ ]:
# # If there are mismatches, inspect the raw country names in UNICEF for near-matches
# if missing_from_unicef:
#     all_unicef_countries = set(unicef_long["country"].dropna().unique())
#     print("All UNICEF European-ish country names (for manual mapping):")
#     # Show any that look plausibly European
#     eu_adjacent = [c for c in sorted(all_unicef_countries)
#                    if any(m[:4].lower() in c.lower() for m in missing_from_unicef)]
#     for c in eu_adjacent:
#         print(f"  '{c}'")

All UNICEF European-ish country names (for manual mapping):
  'Czechia'
  'Netherlands (Kingdom of the)'


In [ ]:
# Manual country name mapping (fill in if mismatches found above)
# Example: 
COUNTRY_MAP = {
    "Czechia"                       : "Czech Republic",
    "Netherlands (Kingdom of the)"  :  "Netherlands"
}

if COUNTRY_MAP:
    unicef_eu["country"] = unicef_eu["country"].replace(COUNTRY_MAP)
    print(f"Applied {len(COUNTRY_MAP)} country name remappings.")
else:
    print("No country name remapping needed.")

Applied 2 country name remappings.


In [75]:
# Drop rows with no coverage value (fully null years)
before = len(unicef_eu)
unicef_eu = unicef_eu.dropna(subset=["coverage_pct"])
print(f"Dropped {before - len(unicef_eu)} fully null coverage rows.")
print(f"Final UNICEF EU shape: {unicef_eu.shape}")

Dropped 1248 fully null coverage rows.
Final UNICEF EU shape: (5760, 6)


In [76]:
# Save cleaned UNICEF data
unicef_out_dir = CLEANED_DIR / "UNICEF_Immunization"
unicef_out_dir.mkdir(parents=True, exist_ok=True)

unicef_out_path = unicef_out_dir / "unicef_immunization_eu_cleaned.csv"
unicef_eu.to_csv(unicef_out_path, index=False, encoding="utf-8")
print(f"Saved: {unicef_out_path.name}  ({len(unicef_eu)} rows)")

Saved: unicef_immunization_eu_cleaned.csv  (5760 rows)


---
## Summary

In [ ]:
print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(f"\n{'LGBT EU Survey':}")
for key, df in lgbt_cleaned.items():
    short = key.replace("LGBT_Survey_", "")
    print(f"  {short:<35}  {len(df):>6} rows  {len(df.columns)} cols")

print(f"\n{'HIV/AIDS — Schema A (no_of_*)':}")
for key, df in schema_a_cleaned.items():
    print(f"  {key:<45}  {len(df):>4} rows  {len(df.columns)} cols")

print(f"\n{'HIV/AIDS — Schema B (art_*)':}")
for key, df in schema_b_cleaned.items():
    print(f"  {key:<45}  {len(df):>4} rows  {len(df.columns)} cols")

print(f"\n{'UNICEF Immunization':}")
print(f"  {'unicef_immunization_eu_cleaned':<45}  {len(unicef_eu):>4} rows  {len(unicef_eu.columns)} cols")

print(f"\n{'EU country list':}  {len(EU_COUNTRIES)} countries")
print(f"\n✅  All cleaned files saved to: {CLEANED_DIR}")
# print("\n➡️  Next step: 7_generate_text_knowledge_base.ipynb")

CLEANING SUMMARY

LGBT EU Survey
  DailyLife                             32836 rows  6 cols
  Discrimination                        15228 rows  6 cols
  RightsAwareness                        3640 rows  6 cols
  TransgenderSpecificQuestions           3301 rows  6 cols
  ViolenceAndHarassment                 43633 rows  6 cols

HIV/AIDS — Schema A (no_of_*)
  no_of_cases_adults_15_to_49_by_country_clean    104 rows  6 cols
  no_of_deaths_by_country_clean                    78 rows  6 cols
  no_of_people_living_with_hiv_by_country_clean   104 rows  6 cols

HIV/AIDS — Schema B (art_*)
  art_coverage_by_country_clean                    26 rows  11 cols
  art_pediatric_coverage_by_country_clean          26 rows  11 cols

UNICEF Immunization
  unicef_immunization_eu_cleaned                 5760 rows  6 cols

EU country list  28 countries

✅  All cleaned files saved to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\2_cleaned


: 